In [62]:
import os
from collections import defaultdict
from itertools import chain

import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
    SingularityVote,
)
from social_groups.reporting.parsing import (
    AnswerOptions,
    AnswerParser,
    AnswerComparer,
)
from social_groups.directories import REPORTING_DIR
from social_groups.reporting.plots.config import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_decision_scheme import calculate_decision_scheme

from social_groups.reporting.heterogenous_comparison.data_retrieval import (
    get_baseline_frame,
    get_mad_frame,
    apply_parsing_and_group_decision,
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
output_dir = REPORTING_DIR / "heterogeneous_group"
os.makedirs(output_dir, exist_ok=True)

triple_underscore_handling = "random"

In [64]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="random"
)

In [65]:
baseline_frame = get_baseline_frame()

In [66]:
baseline_frame

id,run_id,question_id,phoenix_span_id,phoenix_span_url,run_identifier,final_answer,original_question_id,category,question,answer_string,model_name
i64,i64,i64,str,str,str,str,i64,str,str,str,str
0,0,0,"""c9ce50b76ffe168f""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, so the question …",6622,"""health""","""Q: What stable isotope is comm…","""J""","""Qwen/Qwen3-4B"""
1,0,1,"""87b17edffe475e00""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",1190,"""law""","""Q: A company contracted with a…","""E""","""Qwen/Qwen3-4B"""
2,0,2,"""fc4fba7e31996dbd""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",9456,"""physics""","""Q: An electric dipole consisti…","""I""","""Qwen/Qwen3-4B"""
3,0,3,"""a07b8366a728060b""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",5681,"""other""","""Q: In 2018, about how many chi…","""G""","""Qwen/Qwen3-4B"""
4,0,4,"""0d68cb744bad1f9c""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",1945,"""law""","""Q: A wealthy woman often wore …","""E""","""Qwen/Qwen3-4B"""
…,…,…,…,…,…,…,…,…,…,…,…
335,2,98,"""161a98855c25244f""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",7365,"""economics""","""Q: Which of the following exam…","""F""","""Qwen/Qwen3-14B"""
336,2,59,"""286db26b7a2629e7""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",6942,"""economics""","""Q: In the extreme case, $ 1,00…","""B""","""Qwen/Qwen3-14B"""
337,2,58,"""394fd48a69ef5655""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",2048,"""psychology""","""Q: Describe Sherman's and Key'…","""F""","""Qwen/Qwen3-14B"""


In [67]:
print("Number of unparsable answers:")
print(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .group_by("model_name")
    .agg(
        no_null=pl.col(AnalysisColumn.parsed_answer.value)
        .str.starts_with("___")
        .not_()
        .sum(),
        null_percentage=(
            pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
            * 100
        ).round(2),
    )
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

baseline_analysis = (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

baseline_analysis

Number of unparsable answers:
shape: (3, 3)
┌─────────────────┬─────────┬─────────────────┐
│ model_name      ┆ no_null ┆ null_percentage │
│ ---             ┆ ---     ┆ ---             │
│ str             ┆ u32     ┆ f64             │
╞═════════════════╪═════════╪═════════════════╡
│ Qwen/Qwen3-0.6B ┆ 90      ┆ 10.0            │
│ Qwen/Qwen3-4B   ┆ 88      ┆ 12.0            │
│ Qwen/Qwen3-14B  ┆ 93      ┆ 7.0             │
└─────────────────┴─────────┴─────────────────┘


model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.37
"""Qwen/Qwen3-4B""",0.6
"""Qwen/Qwen3-14B""",0.65


## Analzying Basic Group Behaviour

In [68]:
mad_frame = get_mad_frame()
mad_frame

shape: (1_900, 17)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ category   ┆ question   ┆ answer_str ┆ model_name │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ ---        ┆ ---        ┆ ing        ┆ s          │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ str        ┆ str        ┆ ---        ┆ ---        │
│      ┆        ┆             ┆ str        ┆   ┆            ┆            ┆ str        ┆ list[str]  │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 230  ┆ 3      ┆ 63          ┆ bf94cefe4f ┆ … ┆ philosophy ┆ Q: The     ┆ C          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 7d6a67     ┆   ┆            ┆ theory     ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ that says  ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ mental…    ┆            ┆            │
│ 231  ┆ 3      ┆ 6           ┆ 997b3806bf ┆ … ┆ computer   ┆ Q: In      ┆ C          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ b3898e     ┆   ┆ science    ┆ building a ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ linear     ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ regres…    ┆            ┆            │
│ 238  ┆ 3      ┆ 41          ┆ 17efe8d5fd ┆ … ┆ math       ┆ Q: Pat     ┆ I          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 7b7069     ┆   ┆            ┆ bounces a  ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ basketball ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ 25…        ┆            ┆            │
│ 239  ┆ 3      ┆ 105         ┆ 1ab4b05cf3 ┆ … ┆ other      ┆ Q: Which   ┆ I          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 0bf020     ┆   ┆            ┆ of the     ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ following  ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ list…      ┆            ┆            │
│ 248  ┆ 3      ┆ 8           ┆ c385b50db1 ┆ … ┆ chemistry  ┆ Q: A       ┆ E          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 39472a     ┆   ┆            ┆ sample of  ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ bristle    ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ cone pi…   ┆            ┆            │
│ …    ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 2195 ┆ 21     ┆ 10          ┆ c21a7361cd ┆ … ┆ physics    ┆ Q: X rays  ┆ J          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ d9e460     ┆   ┆            ┆ scattered  ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ from rock  ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ …          ┆            ┆ …          │
│ 2196 ┆ 21     ┆ 14          ┆ 95c6302a0f ┆ … ┆ math       ┆ Q: John    ┆ H          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 46a9fb     ┆   ┆            ┆ noticed    ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ that the   ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ angle…     ┆            ┆ …          │
│ 2197 ┆ 21     ┆ 45          ┆ 6228e2254e ┆ … ┆ law        ┆ Q: A busin ┆ B          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ f5fcda     ┆   ┆            ┆ essman is  ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ the owner  ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ …          ┆            ┆ …          │
│ 2198 ┆ 21     ┆ 43          ┆ 2d8bf17996

In [69]:
model_name_sort = {
    "Qwen/Qwen3-14B": 1,
    "Qwen/Qwen3-4B": 2,
    "Qwen/Qwen3-0.6B": 3,
}

mad_analysis = apply_parsing_and_group_decision(
    mad_frame, parser, comparer, group_reply
)

table_page_46 = pl.concat(
    [
        (
            mad_analysis.group_by("group_constellation")
            .agg(accuracy=pl.col("is_correct").mean())
            .with_columns(origin=pl.lit("(MAD)"))
        ),
        (
            baseline_analysis.with_columns(
                group_constellation=pl.col("model_name").replace(
                    MODEL_NAME_TO_LETTER_MAPPING
                ),
                origin=pl.lit("(baseline)"),
            ).drop("model_name")
        ),
    ],
    how="diagonal",
)

table_page_46.write_csv(output_dir / "llm_based_table_page_46.csv")

table_page_46

group_constellation,accuracy,origin
str,f64,str
"""LL""",0.32,"""(MAD)"""
"""HL""",0.57,"""(MAD)"""
"""MM""",0.61,"""(MAD)"""
"""HHM""",0.72,"""(MAD)"""
"""HLL""",0.47,"""(MAD)"""
…,…,…
"""HM""",0.65,"""(MAD)"""
"""LLM""",0.56,"""(MAD)"""
"""L""",0.37,"""(baseline)"""


In [70]:
original_table_page46 = pl.DataFrame(
    {
        "group_constellation": [
            "HHH",
            "HHM",
            "HHL",
            "HML",
            "HMM",
            "H",
            "HLL",
            "MMM",
            "M",
            "MML",
            "MLL",
            "L",
            "LLL",
        ],
        "score (-115 to 115)": [80, 74, 67, 64, 61, 60, 56, 48, 42, 39, 37, 25, 21],
    }
)
original_table_page46.write_csv(output_dir / "human_based_table_page_46.csv")
original_table_page46

group_constellation,score (-115 to 115)
str,i64
"""HHH""",80
"""HHM""",74
"""HHL""",67
"""HML""",64
"""HMM""",61
…,…
"""M""",42
"""MML""",39
"""MLL""",37


In [71]:
decision_schemes = (
    mad_analysis.group_by("group_constellation")
    .map_groups(
        lambda g: calculate_decision_scheme(
            g,
            AnalysisColumn.parsed_individual_answers_before.value,
            AnalysisColumn.parsed_individual_answers_after.value,
            "answer_string",
            group_reply,
            comparer,
        ).select(
            pl.lit(g["group_constellation"].unique().item()).alias(
                "group_constellation"
            ),
            "Correct Members Beginning",
            "correct",
            "incorrect",
        )
    )
    .sort("group_constellation")
)

decision_schemes.write_csv(output_dir / "group_decision_schemes.csv")

decision_schemes

group_constellation,Correct Members Beginning,correct,incorrect
str,u32,f64,f64
"""H""",1,0.944444,0.055556
"""H""",0,0.142857,0.857143
"""HH""",2,1.0,0.0
"""HH""",1,0.6875,0.3125
"""HH""",0,0.181818,0.818182
…,…,…,…
"""MM""",0,0.083333,0.916667
"""MMM""",3,0.890909,0.109091
"""MMM""",2,1.0,0.0


## Unparsable answers:

In [72]:
unparsable_answers_per_model = defaultdict(int)

### In the baseline:

In [73]:
for answer in (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .filter(pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___"))
    .select(["final_answer", "model_name"])
    .iter_rows()
):
    print(answer[1], ": \n")
    print(answer[0][-150:])
    print("-" * 50)
    unparsable_answers_per_model[answer[1]] += 1

Qwen/Qwen3-4B : 

 annual interest rate as (Total Interest / Principal) * (12 / 12) * 100, but that's the same as before.

Alternatively, maybe the problem is using the
--------------------------------------------------
Qwen/Qwen3-4B : 

netic field. The formula for the hyperfine splitting is:

ΔB = (Δf) / (giso)

But if Δf is 500 MHz, then:

ΔB = 500 / 2.12 ≈ 235.8 T. But that's 235,8
--------------------------------------------------
Qwen/Qwen3-4B : 

sec². So, if I have h1 - h2 in Btu/lbm, I need to convert it to ft²/sec². Let me use the following conversion:

1 Btu/lbm = 778.169 ft-lbf/lbm

Then, 
--------------------------------------------------
Qwen/Qwen3-4B : 

ng = T / (r * w * L) 

So, solving for w from shear: w² = T / (r * τ_shear) 

Solving for L from bearing: L = T / (r * τ_bearing * w) 

But since w is
--------------------------------------------------
Qwen/Qwen3-4B : 

. 

Alternatively, maybe the problem is using the value of Cv for CO2 as 25.1 J/mol·K. 

Alternatively

## In the MAD:

In [74]:
for answer in chain(
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_beginning",
        AnalysisColumn.parsed_individual_answers_before.value,
        "model_names",
    )
    .iter_rows(),
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_end",
        AnalysisColumn.parsed_individual_answers_after.value,
        "model_names",
    )
    .iter_rows(),
):
    for i, parsed in enumerate(answer[1]):
        if parsed.startswith("___"):
            unparsable_answers_per_model[answer[2][i]] += 1
            print(parsed, f"from {answer[2][i]}: \n")
            print(answer[0][i][-150:])
            print("-" * 50)

___not_parsable___ from Qwen/Qwen3-14B: 

 Converting to ft²/sec², since 1 ft·lbf = 32.174 ft²/sec², so R = 85.78 * 32.174 ≈ 2760 ft²/sec²·°R.

Temperature T1 is 600°F, which is 600 + 459.67 =
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-14B: 

 planes. Then d = a / sqrt(2) ≈ 6.29 / 1.414 ≈ 4.45 Å. Then:

sinθ = 0.248 / (2 * 4.45) ≈ 0.248 / 8.9 ≈ 0.0279

θ ≈ 1.6 degrees. Still not matching.


--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-14B: 

ħ^2) / (μ_0 * 4π * r^3) * something. But without knowing r, this is not helpful.

Alternatively, the hyperfine splitting in terms of magnetic field is
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-14B: 

, according to the derivation, they are the same. Therefore, the key is designed based on shear stress. 

So the options that satisfy τ ≤ 11,000 are B
--------------------------------------------------
___not_parsable___ from 

-> Qwen 0.6B often says "Answer should be A /think. The Answer is G"

In [75]:
unparsable_answers_per_model

defaultdict(int,
            {'Qwen/Qwen3-4B': 161,
             'Qwen/Qwen3-0.6B': 371,
             'Qwen/Qwen3-14B': 145})

### Build an overview of parsing error Influence

In [76]:
baseline_frame_with_group_constellation = baseline_frame.with_columns(
    group_constellation=pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING)
    + pl.lit(" (Baseline)")
).drop("model_name")

parsing_error_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for handling in ["null", "wrong", "random"]:
    new_comparer = AnswerComparer(AnswerOptions.letters_A_to_J, handling)

    new_mad = (
        apply_parsing_and_group_decision(mad_frame, parser, new_comparer, group_reply)
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    new_base = (
        baseline_frame_with_group_constellation.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            is_correct=new_comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            )
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    parsing_error_influence_table = parsing_error_influence_table.join(
        pl.concat([new_mad, new_base], how="diagonal").select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({handling})"),
        ),
        on="group_constellation",
        how="inner",
    )

parsing_error_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_constellation,Accuracy (null),Accuracy (wrong),Accuracy (random),deviation
str,f64,f64,f64,f64
"""HM""",0.684783,0.63,0.64,0.054783
"""LM""",0.609195,0.53,0.56,0.079195
"""HHL""",0.663265,0.65,0.65,0.013265
"""LMM""",0.646465,0.64,0.64,0.006465
"""HLL""",0.474747,0.47,0.47,0.004747
…,…,…,…,…
"""H""",0.734694,0.72,0.72,0.014694
"""HHH""",0.760417,0.73,0.73,0.030417
"""M (Baseline)""",0.670455,0.59,0.6,0.080455


-> Using (null) is the best, as unparsed values increase your score... should not be used

-> Using wrong is the worst, could possibly be used to be fair, as it is "not correct"

-> Papers and Benchmarks often use "random", which increases the values artificially and introduces noise.. I do not like it, but to be fair one should use it.

---

-> But in all of out cases, even absolute deviation is actually pretty low. (it gets mitigated in group decisions, as unparsable values are ignored in aggregation)

# Agreeableness -> Number of Cases where they reach conslusion

In [77]:
unanimity = (
    mad_analysis.with_columns(
        pl.col(AnalysisColumn.parsed_individual_answers_after.value)
        .list.n_unique()
        .eq(1)
        .alias("End unanimity"),
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.n_unique()
        .eq(1)
        .alias("Start unanimity"),
    )
    .group_by("group_constellation")
    .agg(
        pl.col("End unanimity").mean(),
        pl.col("Start unanimity").mean(),
        pl.when(pl.col("Start unanimity"))
        .then(None)
        .otherwise(pl.col("End unanimity"))
        .mean()
        .alias("End Unanimity | not Start Unanimity"),
        pl.when(pl.col("Start unanimity"))
        .then(pl.col("End unanimity"))
        .otherwise(None)
        .mean()
        .alias("End Unanimity | Start Unanimity"),
        accuracy=pl.col("is_correct").mean(),
    )
)

unanimity

group_constellation,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy
str,f64,f64,f64,f64,f64
"""H""",1.0,1.0,null,1.0,0.72
"""LLM""",0.74,0.23,0.701299,0.869565,0.56
"""LM""",0.8,0.32,0.794118,0.8125,0.54
"""HHL""",0.76,0.3,0.685714,0.933333,0.65
"""L""",1.0,1.0,null,1.0,0.26
…,…,…,…,…,…
"""HHH""",0.95,0.75,0.84,0.986667,0.73
"""MM""",0.93,0.88,0.916667,0.931818,0.61
"""HM""",0.91,0.75,0.88,0.92,0.65


-> In Human Groups (see Group Problem Solving) there is the tendency that the smarter the group, the more it is a "Truth Supported" Decision Scheme, the "dumber" the group, the more it is "proportional"

### Correlation for groups (ignoring 1 member, as it is always 1)

In [78]:
print("Pearson Correlation:")
print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .corr()
)

print("Spearman Rank Correlation:")

print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Pearson Correlation:
shape: (5, 5)
┌───────────────┬─────────────────┬─────────────────────┬───────────────────────┬──────────┐
│ End unanimity ┆ Start unanimity ┆ End Unanimity | not ┆ End Unanimity | Start ┆ accuracy │
│ ---           ┆ ---             ┆ Start Unan…         ┆ Unanimit…             ┆ ---      │
│ f64           ┆ f64             ┆ ---                 ┆ ---                   ┆ f64      │
│               ┆                 ┆ f64                 ┆ f64                   ┆          │
╞═══════════════╪═════════════════╪═════════════════════╪═══════════════════════╪══════════╡
│ 1.0           ┆ 0.798546        ┆ 0.886054            ┆ 0.429095              ┆ 0.39612  │
│ 0.798546      ┆ 1.0             ┆ 0.63738             ┆ 0.468368              ┆ 0.335234 │
│ 0.886054      ┆ 0.63738         ┆ 1.0                 ┆ 0.031864              ┆ 0.327081 │
│ 0.429095      ┆ 0.468368        ┆ 0.031864            ┆ 1.0                   ┆ 0.329198 │
│ 0.39612       ┆ 0.335234        ┆

-> significantly correlated

---
-> Accuracy Correlates with Unanimity
Of course this could be that if everyone is correct in the beginning, then if they that way, then the chance of being correct is higher,
But can we increase the accuracy by accepting when a group starts with one answer?

## Analysis: Is the group finding answers "together" ? (e.g. can it find answers out of wrong start)

In [79]:
correctness_influence = (
    mad_analysis.with_columns(
        correct_before=comparer(
            pl.col(AnalysisColumn.parsed_combined_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("group_constellation")
    .agg(
        pl.when(pl.col("correct_before"))
        .then(pl.col("is_correct"))
        .otherwise(None)
        .mean()
        .alias("Correct | Correct in Beginning"),
        pl.when(pl.col("correct_before"))
        .then(None)
        .otherwise(pl.col("is_correct"))
        .mean()
        .alias("Correct | !Correct in Beginning"),
        accuracy=pl.col("is_correct").mean(),
    )
)

correctness_influence

group_constellation,Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy
str,f64,f64,f64
"""LLM""",0.833333,0.307692,0.56
"""HL""",0.904762,0.327586,0.57
"""HHH""",0.958333,0.142857,0.73
"""H""",0.944444,0.142857,0.72
"""HLM""",0.936508,0.216216,0.67
…,…,…,…
"""HLL""",0.880952,0.172414,0.47
"""LLL""",0.805556,0.1875,0.41
"""HMM""",0.873016,0.081081,0.58


While the difference in Correct | Correct in Beginning is negligible, the true power lies in changing the answer when They are not correct.

In [80]:
(
    correctness_influence.join(unanimity, how="left", on="group_constellation")
    .filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy_right
f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.16777,0.875644,0.556292,0.417098,0.382353,0.384106,0.875644
-0.16777,1.0,-0.035346,-0.307806,-0.379794,-0.309051,-0.019882,-0.035346
0.875644,-0.035346,1.0,0.57732,0.365782,0.367918,0.469809,1.0
0.556292,-0.307806,0.57732,1.0,0.886432,0.665195,0.578792,0.57732
0.417098,-0.379794,0.365782,0.886432,1.0,0.642596,0.51475,0.365782
0.382353,-0.309051,0.367918,0.665195,0.642596,1.0,-0.060339,0.367918
0.384106,-0.019882,0.469809,0.578792,0.51475,-0.060339,1.0,0.469809
0.875644,-0.035346,1.0,0.57732,0.365782,0.367918,0.469809,1.0


## Influence of Group Aggregation Protocol

In [81]:
group_reply_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for strategy in [MajorityVote(), SingularityVote()]:
    new_group_reply_agg = GroupReplyAggregator(strategy)
    group_reply_influence_table = group_reply_influence_table.join(
        apply_parsing_and_group_decision(
            mad_frame, parser, comparer, new_group_reply_agg
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
        .select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({strategy.__class__.__name__})"),
        ),
        on="group_constellation",
        how="inner",
    )

group_reply_influence_table = group_reply_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_reply_influence_table

group_constellation,Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
str,f64,f64,f64
"""LLL""",0.41,0.4,0.01
"""HMM""",0.58,0.55,0.03
"""HHL""",0.65,0.57,0.08
"""L""",0.25,0.25,0.0
"""LLM""",0.56,0.52,0.04
…,…,…,…
"""HL""",0.58,0.59,0.01
"""HM""",0.63,0.63,0.0
"""LMM""",0.65,0.6,0.05


In [82]:
group_reply_influence_table.drop("group_constellation").with_columns(
    pl.all().rank()
).corr()

Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
f64,f64,f64
1.0,0.962522,0.248674
0.962522,1.0,0.055508
0.248674,0.055508,1.0


-> Slightly Negative Correlation between Accuracy and the deviation, meaning the better the more MajorityVote == SingularityVote -> Same argument as before

## Inter-Model Correctness Correlation (Aka answer diversity)

In [83]:
mad_analysis

shape: (1_900, 23)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ group_cons ┆ ___parsed_ ┆ ___parsed_ ┆ is_correct │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ tellation  ┆ combined_a ┆ combined_a ┆ ---        │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ ---        ┆ nswers_bef ┆ nswers_aft ┆ bool       │
│      ┆        ┆             ┆ str        ┆   ┆ str        ┆ …          ┆ …          ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ ---        ┆ ---        ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ str        ┆ str        ┆            │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 230  ┆ 3      ┆ 63          ┆ bf94cefe4f ┆ … ┆ H          ┆ C          ┆ C          ┆ true       │
│      ┆        ┆             ┆ 7d6a67     ┆   ┆            ┆            ┆            ┆            │
│ 231  ┆ 3      ┆ 6           ┆ 997b3806bf ┆ … ┆ H          ┆ D          ┆ D          ┆ false      │
│      ┆        ┆             ┆ b3898e     ┆   ┆            ┆            ┆            ┆            │
│ 238  ┆ 3      ┆ 41          ┆ 17efe8d5fd ┆ … ┆ H          ┆ I          ┆ I          ┆ true       │
│      ┆        ┆             ┆ 7b7069     ┆   ┆            ┆            ┆            ┆            │
│ 239  ┆ 3      ┆ 105         ┆ 1ab4b05cf3 ┆ … ┆ H          ┆ I          ┆ I          ┆ true       │
│      ┆        ┆             ┆ 0bf020     ┆   ┆            ┆            ┆            ┆            │
│ 248  ┆ 3      ┆ 8           ┆ c385b50db1 ┆ … ┆ H          ┆ D          ┆ D          ┆ false      │
│      ┆        ┆             ┆ 39472a     ┆   ┆            ┆            ┆            ┆            │
│ …    ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 2195 ┆ 21     ┆ 10          ┆ c21a7361cd ┆ … ┆ LMM        ┆ ___differe ┆ A          ┆ false      │
│      ┆        ┆             ┆ d9e460     ┆   ┆            ┆ nt_votes__ ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ _          ┆            ┆            │
│ 2196 ┆ 21     ┆ 14          ┆ 95c6302a0f ┆ … ┆ LMM        ┆ H          ┆ H          ┆ true       │
│      ┆        ┆             ┆ 46a9fb     ┆   ┆            ┆            ┆            ┆            │
│ 2197 ┆ 21     ┆ 45          ┆ 6228e2254e ┆ … ┆ LMM        ┆ E          ┆ E          ┆ false      │
│      ┆        ┆             ┆ f5fcda     ┆   ┆            ┆            ┆            ┆            │
│ 2198 ┆ 21     ┆ 43          ┆ 2d8bf17996 ┆ … ┆ LMM        ┆ H          ┆ H          ┆ true       │
│      ┆        ┆             ┆ c57d3a     ┆   ┆            ┆            ┆            ┆            │
│ 2199 ┆ 21     ┆ 96          ┆ e94f194c52 ┆ … ┆ LMM        ┆ G          ┆ G          ┆ false      │
│      ┆        ┆             ┆ 3dfa5f     ┆   ┆            ┆            ┆            ┆            │
└──────┴────────┴─────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

#### For Individual answers

In [84]:
individual_models_answer_per_question = (
    (
        baseline_frame.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING),
            is_correct=comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            ),
        )
        .pivot(
            values="is_correct",
            index="question_id",
            on="model_name",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .select("question_id", "L", "M", "H")
    .with_columns(pl.exclude("question_id").cast(bool))
)

individual_models_answer_per_question

question_id,L,M,H
i64,bool,bool,bool
0,true,true,true
1,true,false,false
2,true,true,true
3,false,true,true
4,false,false,true
…,…,…,…
115,false,false,false
116,false,false,true
117,false,true,true


In [85]:
individual_models_answer_per_question.drop("question_id").corr()

L,M,H
f64,f64,f64
1.0,0.371816,0.230769
0.371816,1.0,0.530859
0.230769,0.530859,1.0


--> Actually surprisingly different

In [86]:
df = pl.concat(
    [
        individual_models_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
)

df_single = df.select("Given", "L", "M", "H")

df_single

Given,L,M,H
str,f64,f64,f64
"""L_incorrect""",0.0,0.476923,0.569231
"""L_correct""",1.0,0.857143,0.8
"""M_correct""",0.491803,1.0,0.852459
"""M_incorrect""",0.128205,0.0,0.333333
"""H_correct""",0.430769,0.8,1.0
"""H_incorrect""",0.2,0.257143,0.0


-> They are not completely overlapping in the single answer case.

This means could be some diversity effect going on

---

In [87]:
individual_groups_answer_per_question = (
    (
        mad_analysis.filter(pl.col("group_constellation").str.len_chars() == 1).pivot(
            values="is_correct",
            index="question_id",
            on="group_constellation",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .with_columns(pl.exclude("question_id").cast(bool))
)

df_single_mad = pl.concat(
    [
        individual_groups_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", "L", "M", "H")

df_single_mad

Given,L,M,H
str,f64,f64,f64
"""L_correct""",1.0,0.884615,0.923077
"""L_incorrect""",0.0,0.378378,0.648649
"""M_correct""",0.45098,1.0,0.921569
"""M_incorrect""",0.061224,0.0,0.510204
"""H_correct""",0.333333,0.652778,1.0
"""H_incorrect""",0.071429,0.142857,0.0


In [88]:
combined_ind = individual_models_answer_per_question.join(
    individual_groups_answer_per_question, on="question_id", suffix="_mad"
)
pl.concat(
    [
        combined_ind.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in combined_ind.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", pl.exclude("Given"))

Given,L,M,H,H_mad,M_mad,L_mad
str,f64,f64,f64,f64,f64,f64
"""L_correct""",1.0,0.857143,0.8,0.828571,0.685714,0.571429
"""L_incorrect""",0.0,0.476923,0.569231,0.661538,0.415385,0.092308
"""M_correct""",0.491803,1.0,0.852459,0.868852,0.754098,0.409836
"""M_incorrect""",0.128205,0.0,0.333333,0.487179,0.128205,0.025641
"""H_correct""",0.430769,0.8,1.0,0.892308,0.661538,0.353846
…,…,…,…,…,…,…
"""H_mad_correct""",0.402778,0.736111,0.805556,1.0,0.652778,0.333333
"""M_mad_incorrect""",0.22449,0.306122,0.44898,0.510204,0.0,0.061224
"""M_mad_correct""",0.470588,0.901961,0.843137,0.921569,1.0,0.45098


> See Obsidian

#### Is the "nobody is right case" in HHH actually unanimous?

In [89]:
print("Answers where everyone was wrong:")

everyone_wrong = (
    mad_analysis.filter(pl.col("group_constellation") == "HHH")
    .explode(AnalysisColumn.parsed_individual_answers_before.value)
    .with_columns(
        is_individually_correct=comparer(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("question_id")
    .agg(
        number_wrong=pl.len(),
        answers=pl.col(AnalysisColumn.parsed_individual_answers_before.value).implode(),
        is_individually_correct=pl.col("is_individually_correct").implode(),
    )
    .filter(pl.col("is_individually_correct").list.eval(pl.element().not_()).list.all())
    .drop("number_wrong", "is_individually_correct")
)

print(everyone_wrong)
unanimous = (
    everyone_wrong["answers"]
    .filter(everyone_wrong["answers"].list.n_unique() == 1)
    .count()
)
print(f"Of that unanimous: {unanimous}")
print("Contentious:")
print(everyone_wrong["answers"].filter(everyone_wrong["answers"].list.n_unique() != 1))

Answers where everyone was wrong:
shape: (21, 2)
┌─────────────┬─────────────────────────────────┐
│ question_id ┆ answers                         │
│ ---         ┆ ---                             │
│ i64         ┆ list[str]                       │
╞═════════════╪═════════════════════════════════╡
│ 8           ┆ ["D", "D", "D"]                 │
│ 27          ┆ ["E", "E", "E"]                 │
│ 71          ┆ ["H", "H", "H"]                 │
│ 45          ┆ ["C", "C", "C"]                 │
│ 1           ┆ ["C", "C", "C"]                 │
│ …           ┆ …                               │
│ 46          ┆ ["D", "H", "D"]                 │
│ 11          ┆ ["___not_parsable___", "F", "_… │
│ 6           ┆ ["D", "D", "D"]                 │
│ 115         ┆ ["C", "C", "C"]                 │
│ 87          ┆ ["___not_parsable___", "___not… │
└─────────────┴─────────────────────────────────┘
Of that unanimous: 14
Contentious:
shape: (7,)
Series: 'answers' [list[str]]
[
	["B", "___not_parsabl

C# Intra-Group Correctness Correlation (Aka group diversity)